# Inteligencia de Negocio para una Panadería
## Análisis de la Cesta de Mercado y Optimización del Negocio

Este proyecto analiza datos transaccionales de **The Bread Basket**, una panadería/cafetería en Edimburgo, para identificar oportunidades accionables para un negocio de retail alimentario.

### Preguntas de negocio

1. ¿Qué productos generan la mayor parte de la demanda?
2. ¿Cuándo experimenta la panadería su mayor volumen de transacciones?
3. ¿Qué tamaño tienen las cestas de los clientes?
4. ¿Qué productos se compran juntos con más frecuencia de la esperada por azar?
5. ¿Dónde están las mayores oportunidades de venta cruzada, dotación de personal y optimización del surtido?

### Dataset

El dataset público contiene transacciones de la panadería registradas entre finales de 2016 y principios de 2017.

Columnas originales:

- `Date`
- `Time`
- `Transaction`
- `Item`

La versión original contiene registros marcador `NONE`, que se eliminan durante la limpieza.

### Flujo analítico

**Transacciones en bruto → Limpieza → EDA → Construcción de cestas → Reglas de asociación → Recomendaciones de negocio**

> Limitación importante: el dataset **no** contiene precios, márgenes, costes, niveles de stock ni desperdicio. Por lo tanto, este proyecto identifica oportunidades comerciales y operativas, pero no afirma un aumento directo de beneficios.


## 1. Importaciones

In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Carga del dataset público

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/prasertcbs/basic-dataset/master/BreadBasket_DMS.csv"

df_raw = pd.read_csv(DATA_URL)

print(f"Filas en bruto: {len(df_raw):,}")
display(df_raw.head())

## 3. Limpieza e ingeniería de variables

In [ ]:
df = df_raw.copy()

# Estandarizar el texto y eliminar registros marcador
df["Item"] = df["Item"].astype(str).str.strip()
df = df[df["Item"].str.upper() != "NONE"].copy()

# Construir la marca de tiempo
df["datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce",
)

df = df.dropna(subset=["datetime", "Transaction", "Item"])

df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.day_name()
df["is_weekend"] = df["datetime"].dt.dayofweek >= 5

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

print(f"Líneas de venta válidas: {len(df):,}")
print(f"Transacciones únicas: {df['Transaction'].nunique():,}")
print(f"Productos únicos: {df['Item'].nunique():,}")
print(f"Rango de fechas: {df['datetime'].min()} → {df['datetime'].max()}")

## 4. Demanda de productos

In [ ]:
product_units = df["Item"].value_counts()
product_share = product_units / product_units.sum()

top_products = pd.DataFrame({
    "unidades": product_units,
    "porcentaje": product_share,
}).head(15)

display(top_products.style.format({"porcentaje": "{:.1%}"}))

In [ ]:
top10 = top_products.head(10).sort_values("unidades")

plt.figure(figsize=(9, 6))
plt.barh(top10.index, top10["unidades"])
plt.xlabel("Unidades registradas")
plt.ylabel("")
plt.title("Productos con mayor volumen de venta")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_top_products.png", dpi=200, bbox_inches="tight")
plt.show()

### Cola larga del surtido

Un producto con baja frecuencia histórica **no es automáticamente poco rentable**.  
Sin embargo, una cola muy larga de referencias (SKU) de bajo volumen es una señal útil para una revisión del surtido cuando se combina con futuros datos de margen, mermas y requisitos de almacenamiento.

In [ ]:
low_frequency = (product_share < 0.01).sum()

print(
    f"{low_frequency} de {len(product_share)} productos "
    f"representan individualmente menos del 1% del volumen de artículos registrado."
)

## 5. Análisis del tamaño de la cesta

In [ ]:
# Productos únicos por transacción.
# Para el análisis de cesta, las unidades repetidas del mismo producto cuentan una sola vez.
basket_size = (
    df.groupby("Transaction")["Item"]
      .nunique()
      .rename("basket_size")
)

single_item_share = (basket_size == 1).mean()

print(f"Promedio de productos únicos por cesta: {basket_size.mean():.2f}")
print(f"Tamaño mediano de la cesta: {basket_size.median():.0f}")
print(f"Cestas de un solo producto: {single_item_share:.1%}")
print(f"Cestas con <= 5 productos únicos: {(basket_size <= 5).mean():.1%}")

In [ ]:
basket_dist = basket_size.value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(basket_dist.index.astype(str), basket_dist.values)
plt.xlabel("Número de productos distintos por ticket")
plt.ylabel("Número de tickets")
plt.title("Distribución del tamaño de la cesta")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_basket_size.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Demanda temporal

Dos vistas resultan útiles:

- **Transacciones por hora** → carga de trabajo y dotación de personal.
- **Media de transacciones por día operativo, según el día de la semana** → planificación específica por día.

Se usa el número de transacciones en lugar de las líneas de venta en bruto, para que las cestas grandes no inflen artificialmente las estimaciones de flujo de clientes.

In [ ]:
# Una marca de tiempo por ticket
tickets = (
    df.groupby("Transaction", as_index=False)
      .agg(datetime=("datetime", "min"))
)

tickets["date"] = tickets["datetime"].dt.date
tickets["hour"] = tickets["datetime"].dt.hour
tickets["weekday"] = tickets["datetime"].dt.day_name()

hourly_transactions = tickets.groupby("hour").size()

plt.figure(figsize=(9, 5))
plt.bar(hourly_transactions.index, hourly_transactions.values)
plt.xlabel("Hora del día")
plt.ylabel("Tickets")
plt.title("Distribución de transacciones por hora")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "03_transactions_by_hour.png", dpi=200, bbox_inches="tight")
plt.show()

peak_hour = hourly_transactions.idxmax()
print(f"Hora pico de transacciones: {peak_hour}:00")

In [ ]:
daily = (
    tickets.groupby(["date", "weekday"])
           .size()
           .rename("transactions")
           .reset_index()
)

weekday_avg = (
    daily.groupby("weekday")["transactions"]
         .mean()
         .reindex(weekday_order)
)

# Traducción solo para presentación; el agrupado interno usa los nombres en inglés
# que devuelve pandas (dt.day_name()).
weekday_es = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo",
}

plt.figure(figsize=(9, 5))
plt.bar([weekday_es[d] for d in weekday_avg.index], weekday_avg.values)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Media de tickets por día")
plt.title("Demanda media por día de la semana")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_weekday_demand.png", dpi=200, bbox_inches="tight")
plt.show()

print("Día de la semana con mayor promedio:", weekday_es[weekday_avg.idxmax()])
print("Día de la semana con menor promedio:", weekday_es[weekday_avg.idxmin()])

## 7. Análisis de la Cesta de Mercado

Métricas de reglas de asociación:

- **Soporte**: proporción de transacciones que contienen ambos productos.
- **Confianza A → B**: probabilidad de que esté presente B cuando está presente A.
- **Lift**: cuánto más a menudo aparecen A y B juntos de lo esperado si fueran independientes.

Un lift superior a 1 indica una asociación positiva. Este notebook se centra en reglas con suficiente soporte para ser comercialmente relevantes, en lugar de coincidencias poco frecuentes.

In [ ]:
# Convertir las transacciones en conjuntos de artículos únicos
basket_series = (
    df.groupby("Transaction")["Item"]
      .apply(lambda x: frozenset(x))
)

n_transactions = len(basket_series)

item_transaction_count = Counter()
pair_count = Counter()

for basket in basket_series:
    for item in basket:
        item_transaction_count[item] += 1
    for pair in combinations(sorted(basket), 2):
        pair_count[pair] += 1

item_support = {
    item: count / n_transactions
    for item, count in item_transaction_count.items()
}

rules = []

for (a, b), count_ab in pair_count.items():
    support_ab = count_ab / n_transactions

    confidence_a_b = support_ab / item_support[a]
    confidence_b_a = support_ab / item_support[b]

    lift = support_ab / (item_support[a] * item_support[b])

    rules.append({
        "antecedente": a,
        "consecuente": b,
        "soporte": support_ab,
        "confianza": confidence_a_b,
        "lift": lift,
        "coocurrencias": count_ab,
    })

    rules.append({
        "antecedente": b,
        "consecuente": a,
        "soporte": support_ab,
        "confianza": confidence_b_a,
        "lift": lift,
        "coocurrencias": count_ab,
    })

rules_df = pd.DataFrame(rules)

# Conservar reglas con al menos 1% de soporte y asociación positiva.
business_rules = (
    rules_df[
        (rules_df["soporte"] >= 0.01)
        & (rules_df["lift"] >= 1.20)
    ]
    .sort_values(["confianza", "lift"], ascending=False)
    .reset_index(drop=True)
)

display(
    business_rules.head(20).style.format({
        "soporte": "{:.2%}",
        "confianza": "{:.2%}",
        "lift": "{:.2f}",
    })
)

In [ ]:
top_rules = business_rules.head(10).copy()
top_rules["regla"] = top_rules["antecedente"] + " → " + top_rules["consecuente"]
top_rules = top_rules.sort_values("confianza")

plt.figure(figsize=(10, 6))
plt.barh(top_rules["regla"], top_rules["confianza"])
plt.xlabel("Confianza")
plt.xlim(0, 1)
plt.title("Reglas de asociación con mayor confianza")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_association_rules.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Diagnóstico de oportunidades de negocio

In [ ]:
coffee_support = item_support.get("Coffee", np.nan)

peak_start = 9
peak_end = 14
peak_share = tickets["hour"].between(peak_start, peak_end).mean()

summary = pd.DataFrame({
    "Métrica": [
        "Transacciones",
        "Productos",
        "Cestas de un solo producto",
        "Soporte de transacciones con café",
        f"Transacciones entre las {peak_start}:00 y las {peak_end}:59",
        "SKUs de baja frecuencia (<1% del volumen de artículos)",
    ],
    "Valor": [
        f"{n_transactions:,}",
        f"{df['Item'].nunique():,}",
        f"{single_item_share:.1%}",
        f"{coffee_support:.1%}",
        f"{peak_share:.1%}",
        f"{low_frequency}/{len(product_share)}",
    ],
})

display(summary)

## 9. Conclusiones de negocio

El código anterior recalcula las conclusiones directamente a partir del dataset público.

### 1. La venta cruzada debería centrarse en las cestas de un solo producto

Una parte sustancial de las transacciones contiene un único producto. Esto crea una **oportunidad medible de ampliación de la cesta**.

**Acción de negocio:**  
Probar sugerencias contextuales de productos adicionales en lugar de descuentos generales. Por ejemplo, tras seleccionar una bebida, recomendar un producto de panadería complementario respaldado por evidencia de reglas de asociación.

**Cómo validarlo:**  
Test A/B de la tasa de conversión, el tamaño medio de la cesta y el margen bruto por transacción.

---

### 2. El café actúa como producto ancla

El café tiene un soporte transaccional muy alto y aparece repetidamente en reglas de asociación fuertes.

**Acción de negocio:**  
Tratar la disponibilidad del café y la velocidad de servicio como prioridades operativas. Colocar productos complementarios de alto valor cerca del flujo de compra del café.

**Importante:**  
Una alta frecuencia de compra conjunta no demuestra que sea necesario un descuento. Si los clientes ya compran una combinación de forma natural, aplicar un descuento puede simplemente reducir el margen.

---

### 3. Las reglas de asociación pueden impulsar paquetes dirigidos

Las reglas con soporte relevante y lift superior a 1 son mejores candidatas para la venta cruzada que simplemente combinar los dos productos más populares.

**Acción de negocio:**  
Usar reglas como `Toast → Coffee` u otras combinaciones de alto lift para diseñar:

- recomendaciones en el punto de venta (POS)
- sugerencias en el menú
- avisos en el pedido digital
- experimentos de paquetes limitados

Medir la conversión incremental antes de hacerlos permanentes.

---

### 4. La dotación de personal y la producción deberían seguir la demanda horaria

El volumen de transacciones se concentra en horas específicas.

**Acción de negocio:**  

- programar más personal de atención al cliente en torno al pico observado;
- preparar los productos de mayor rotación poco antes de que aumente la demanda;
- aprovechar los periodos de menor actividad para reponer stock, limpiar y preparar.

Esto es optimización operativa basada en el flujo de transacciones, no solo en las ventas totales.

---

### 5. La demanda según el día de la semana debería influir en la planificación

El volumen medio diario de transacciones varía según el día de la semana.

**Acción de negocio:**  
Crear plantillas de producción y personal específicas para cada día de la semana en lugar de usar el mismo plan todos los días.

Los días de menor demanda también son útiles para probar promociones sin afectar a la operación en horas punta.

---

### 6. La cola larga de productos merece una revisión del surtido

Muchas referencias (SKU) representan individualmente menos del 1% del volumen de artículos.

**Acción de negocio:**  
**No** eliminarlas automáticamente. Combinar la frecuencia de venta con:

- margen bruto;
- solapamiento de ingredientes;
- tiempo de preparación;
- vida útil;
- desperdicio de alimentos;
- valor estratégico/de menú.

Los productos de bajo volumen, bajo margen y alto desperdicio se convierten en fuertes candidatos a racionalización.

---

## Lo que este dataset no puede decirnos

Sin datos de precio, margen, stock y desperdicio, este análisis no puede estimar de forma legítima:

- el aumento de beneficios;
- el precio óptimo;
- la reducción de desperdicio en euros;
- las cantidades óptimas de reposición de inventario;
- el retorno de la inversión (ROI) de las promociones.

Eso requeriría datos de negocio adicionales.

Esta limitación es útil en un portafolio profesional: una buena Ciencia de Datos distingue entre evidencia y suposiciones.

## 10. Próxima iteración recomendada

Para un cliente real de panadería, la siguiente versión de este análisis debería añadir:

1. Precio y coste unitario.
2. Margen del producto.
3. Desperdicio / cantidad no vendida.
4. Inventario y roturas de stock.
5. Historial de promociones.
6. Clima y eventos locales.
7. Identificador de cliente o datos de fidelización.

Eso permitiría que el proyecto evolucionara de **analítica descriptiva + de asociación** hacia:

**Previsión de demanda → planificación de producción → optimización del desperdicio → optimización de la rentabilidad**